# Merge yearly monthly-index features into training dataset

This notebook merges yearly NDVI/NDWI index features from `full_dataset_20m_monthly_with_indices.zarr` into `training_data_with_features.zarr` and writes a new dataset.

Target output: `training_data_with_features_plus_monthly_indices.zarr`

In [1]:
import shutil
import time
from pathlib import Path

import numpy as np
import xarray as xr

TRAIN_PATH = Path("training_data_with_features.zarr")
SOURCE_PATH = Path("full_dataset_20m_monthly_with_indices.zarr")
OUTPUT_PATH = Path("training_data_with_features_plus_monthly_indices.zarr")

FEATURES_TO_MERGE = [
    "ndvi_cv_year",
    "ndvi_max_m2m_drop_year",
    "ndvi_max_year",
    "ndvi_min_year",
    "ndvi_std_year",
    "ndwi_cv_year",
    "ndwi_max_m2m_drop_year",
    "ndwi_max_year",
    "ndwi_min_year",
    "ndwi_std_year",
]

OUTPUT_PIXEL_CHUNK = 250_000
QA_SAMPLE_COUNT = 20

print(f"Training dataset: {TRAIN_PATH}")
print(f"Source dataset  : {SOURCE_PATH}")
print(f"Output dataset  : {OUTPUT_PATH}")
print(f"Features to merge ({len(FEATURES_TO_MERGE)}): {FEATURES_TO_MERGE}")

Training dataset: training_data_with_features.zarr
Source dataset  : full_dataset_20m_monthly_with_indices.zarr
Output dataset  : training_data_with_features_plus_monthly_indices.zarr
Features to merge (10): ['ndvi_cv_year', 'ndvi_max_m2m_drop_year', 'ndvi_max_year', 'ndvi_min_year', 'ndvi_std_year', 'ndwi_cv_year', 'ndwi_max_m2m_drop_year', 'ndwi_max_year', 'ndwi_min_year', 'ndwi_std_year']


In [2]:
def to_string_array(values: np.ndarray) -> np.ndarray:
    array = np.asarray(values)
    if array.dtype.kind in {"S", "a"}:
        return np.char.decode(array, "utf-8")
    if array.dtype.kind == "O":
        return np.array([
            value.decode("utf-8") if isinstance(value, (bytes, bytearray)) else str(value)
            for value in array
        ])
    return array.astype(str)


train_ds = xr.open_zarr(TRAIN_PATH)
source_ds = xr.open_zarr(SOURCE_PATH)

print("TRAIN dims:", dict(train_ds.sizes))
print("SOURCE dims:", dict(source_ds.sizes))

required_train_vars = ["cube_name", "x", "y"]
for var_name in required_train_vars:
    if var_name not in train_ds:
        raise KeyError(f"Missing required variable '{var_name}' in training dataset.")

missing_source_features = [feature for feature in FEATURES_TO_MERGE if feature not in source_ds]
if missing_source_features:
    raise KeyError(f"Missing source features: {missing_source_features}")

for feature in FEATURES_TO_MERGE:
    expected_dims = ("cube", "year", "y", "x")
    actual_dims = tuple(source_ds[feature].dims)
    if actual_dims != expected_dims:
        raise ValueError(
            f"Feature '{feature}' has dims {actual_dims}, expected {expected_dims}."
        )

if not np.array_equal(train_ds["year"].values, source_ds["year"].values):
    raise ValueError(
        "Year coordinates do not match between training and source datasets."
    )

x_idx = train_ds["x"].values.astype(np.int64)
y_idx = train_ds["y"].values.astype(np.int64)

if x_idx.min() < 0 or x_idx.max() >= source_ds.sizes["x"]:
    raise ValueError(
        f"Training x indices out of bounds: min={x_idx.min()}, max={x_idx.max()}, source_x={source_ds.sizes['x']}"
    )

if y_idx.min() < 0 or y_idx.max() >= source_ds.sizes["y"]:
    raise ValueError(
        f"Training y indices out of bounds: min={y_idx.min()}, max={y_idx.max()}, source_y={source_ds.sizes['y']}"
    )

source_cube_names = to_string_array(source_ds["cube"].values)
train_cube_names = to_string_array(train_ds["cube_name"].values)

sorter = np.argsort(source_cube_names)
source_cube_names_sorted = source_cube_names[sorter]
positions = np.searchsorted(source_cube_names_sorted, train_cube_names)

positions_clipped = np.clip(positions, 0, source_cube_names_sorted.size - 1)
matched = source_cube_names_sorted[positions_clipped] == train_cube_names

if not np.all(matched):
    missing_cube_names = np.unique(train_cube_names[~matched])
    sample_missing = missing_cube_names[:10].tolist()
    raise ValueError(
        "Fail-fast: some training cube_name values are not present in source cube coordinates. "
        f"Missing count={missing_cube_names.size}, sample={sample_missing}"
    )

mapped_source_cube_idx = sorter[positions]

pixel_order = np.arange(train_ds.sizes["pixel"], dtype=np.int64)
sort_by_cube = np.argsort(mapped_source_cube_idx, kind="mergesort")
sorted_cube_idx = mapped_source_cube_idx[sort_by_cube]

group_starts = np.flatnonzero(np.r_[True, sorted_cube_idx[1:] != sorted_cube_idx[:-1]])
group_ends = np.r_[group_starts[1:], sorted_cube_idx.size]

cube_groups: list[tuple[int, np.ndarray]] = []
for start, end in zip(group_starts, group_ends):
    source_cube_idx = int(sorted_cube_idx[start])
    pixels_in_cube = sort_by_cube[start:end]
    cube_groups.append((source_cube_idx, pixels_in_cube))

print(f"Cube groups prepared: {len(cube_groups)}")
print(f"Total pixels grouped: {sum(group[1].size for group in cube_groups):,}")

TRAIN dims: {'pixel': 8155205, 'year': 7, 's2_band': 7}
SOURCE dims: {'cube': 2216, 'year': 7, 'y': 128, 'x': 128, 'month': 12}
Cube groups prepared: 1975
Total pixels grouped: 8,155,205


## Execute merge

This cell creates a new output dataset by copying the full training store and appending each new feature one-by-one as `(pixel, year)` with `float32` dtype.

It will fail if the output path already exists.

In [3]:
if OUTPUT_PATH.exists():
    raise FileExistsError(
        f"Output path already exists: {OUTPUT_PATH}. Delete or rename it before running this merge."
    )

copy_start = time.time()
shutil.copytree(TRAIN_PATH, OUTPUT_PATH)
print(f"Copied base training dataset to {OUTPUT_PATH} in {time.time() - copy_start:.1f}s")

n_pixels = train_ds.sizes["pixel"]
n_years = train_ds.sizes["year"]
pixel_coord = train_ds["pixel"].values
year_coord = train_ds["year"].values


def extract_feature_values(feature_name: str) -> np.ndarray:
    source_var = source_ds[feature_name]
    merged_values = np.full((n_pixels, n_years), np.nan, dtype=np.float32)

    total_groups = len(cube_groups)
    feature_start = time.time()

    for group_index, (source_cube_idx, pixels_in_cube) in enumerate(cube_groups, start=1):
        y_select = xr.DataArray(y_idx[pixels_in_cube], dims="pixel_group")
        x_select = xr.DataArray(x_idx[pixels_in_cube], dims="pixel_group")

        source_values = (
            source_var
            .isel(cube=source_cube_idx, y=y_select, x=x_select)
            .transpose("pixel_group", "year")
            .values
        )
        merged_values[pixels_in_cube, :] = source_values.astype(np.float32, copy=False)

        if group_index % 200 == 0 or group_index == total_groups:
            elapsed = time.time() - feature_start
            print(
                f"  {feature_name}: cube groups {group_index}/{total_groups} processed "
                f"(elapsed {elapsed:.1f}s)"
            )

    return merged_values


for feature_name in FEATURES_TO_MERGE:
    print(f"\nMerging feature: {feature_name}")
    start = time.time()

    feature_values = extract_feature_values(feature_name)
    feature_dataset = xr.Dataset(
        data_vars={
            feature_name: (("pixel", "year"), feature_values),
        },
        coords={
            "pixel": pixel_coord,
            "year": year_coord,
        },
    )

    feature_dataset.to_zarr(
        OUTPUT_PATH,
        mode="a",
        encoding={
            feature_name: {
                "dtype": "float32",
                "chunks": (OUTPUT_PIXEL_CHUNK, n_years),
            }
        },
    )

    elapsed = time.time() - start
    print(f"Finished {feature_name} in {elapsed:.1f}s")

print("\nMerge completed.")
print(f"Output written to: {OUTPUT_PATH}")

Copied base training dataset to training_data_with_features_plus_monthly_indices.zarr in 15.8s

Merging feature: ndvi_cv_year
  ndvi_cv_year: cube groups 200/1975 processed (elapsed 4.2s)
  ndvi_cv_year: cube groups 400/1975 processed (elapsed 7.3s)
  ndvi_cv_year: cube groups 600/1975 processed (elapsed 10.9s)
  ndvi_cv_year: cube groups 800/1975 processed (elapsed 14.5s)
  ndvi_cv_year: cube groups 1000/1975 processed (elapsed 18.8s)
  ndvi_cv_year: cube groups 1200/1975 processed (elapsed 23.4s)
  ndvi_cv_year: cube groups 1400/1975 processed (elapsed 28.0s)
  ndvi_cv_year: cube groups 1600/1975 processed (elapsed 32.2s)
  ndvi_cv_year: cube groups 1800/1975 processed (elapsed 37.5s)
  ndvi_cv_year: cube groups 1975/1975 processed (elapsed 42.4s)


c:\Users\bartu\Desktop\Fonda-scikit\venv\Lib\site-packages\zarr\api\asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Finished ndvi_cv_year in 45.9s

Merging feature: ndvi_max_m2m_drop_year
  ndvi_max_m2m_drop_year: cube groups 200/1975 processed (elapsed 5.1s)
  ndvi_max_m2m_drop_year: cube groups 400/1975 processed (elapsed 9.3s)
  ndvi_max_m2m_drop_year: cube groups 600/1975 processed (elapsed 13.7s)
  ndvi_max_m2m_drop_year: cube groups 800/1975 processed (elapsed 18.0s)
  ndvi_max_m2m_drop_year: cube groups 1000/1975 processed (elapsed 24.3s)
  ndvi_max_m2m_drop_year: cube groups 1200/1975 processed (elapsed 30.5s)
  ndvi_max_m2m_drop_year: cube groups 1400/1975 processed (elapsed 35.5s)
  ndvi_max_m2m_drop_year: cube groups 1600/1975 processed (elapsed 39.7s)
  ndvi_max_m2m_drop_year: cube groups 1800/1975 processed (elapsed 43.9s)
  ndvi_max_m2m_drop_year: cube groups 1975/1975 processed (elapsed 47.4s)


c:\Users\bartu\Desktop\Fonda-scikit\venv\Lib\site-packages\zarr\api\asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Finished ndvi_max_m2m_drop_year in 50.9s

Merging feature: ndvi_max_year
  ndvi_max_year: cube groups 200/1975 processed (elapsed 5.9s)
  ndvi_max_year: cube groups 400/1975 processed (elapsed 10.7s)
  ndvi_max_year: cube groups 600/1975 processed (elapsed 16.1s)
  ndvi_max_year: cube groups 800/1975 processed (elapsed 21.4s)
  ndvi_max_year: cube groups 1000/1975 processed (elapsed 28.8s)
  ndvi_max_year: cube groups 1200/1975 processed (elapsed 34.7s)
  ndvi_max_year: cube groups 1400/1975 processed (elapsed 40.5s)
  ndvi_max_year: cube groups 1600/1975 processed (elapsed 44.9s)
  ndvi_max_year: cube groups 1800/1975 processed (elapsed 51.3s)
  ndvi_max_year: cube groups 1975/1975 processed (elapsed 55.8s)


...

c:\Users\bartu\Desktop\Fonda-scikit\venv\Lib\site-packages\zarr\api\asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Finished ndwi_max_year in 43.7s

Merging feature: ndwi_min_year
  ndwi_min_year: cube groups 200/1975 processed (elapsed 4.6s)
  ndwi_min_year: cube groups 400/1975 processed (elapsed 9.5s)
  ndwi_min_year: cube groups 600/1975 processed (elapsed 14.4s)
  ndwi_min_year: cube groups 800/1975 processed (elapsed 18.5s)
  ndwi_min_year: cube groups 1000/1975 processed (elapsed 24.0s)
  ndwi_min_year: cube groups 1200/1975 processed (elapsed 28.4s)
  ndwi_min_year: cube groups 1400/1975 processed (elapsed 32.5s)
  ndwi_min_year: cube groups 1600/1975 processed (elapsed 36.3s)
  ndwi_min_year: cube groups 1800/1975 processed (elapsed 40.0s)
  ndwi_min_year: cube groups 1975/1975 processed (elapsed 43.1s)


c:\Users\bartu\Desktop\Fonda-scikit\venv\Lib\site-packages\zarr\api\asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Finished ndwi_min_year in 46.0s

Merging feature: ndwi_std_year
  ndwi_std_year: cube groups 200/1975 processed (elapsed 4.6s)
  ndwi_std_year: cube groups 400/1975 processed (elapsed 8.5s)
  ndwi_std_year: cube groups 600/1975 processed (elapsed 13.1s)
  ndwi_std_year: cube groups 800/1975 processed (elapsed 17.7s)
  ndwi_std_year: cube groups 1000/1975 processed (elapsed 21.9s)
  ndwi_std_year: cube groups 1200/1975 processed (elapsed 25.6s)
  ndwi_std_year: cube groups 1400/1975 processed (elapsed 29.7s)
  ndwi_std_year: cube groups 1600/1975 processed (elapsed 33.2s)
  ndwi_std_year: cube groups 1800/1975 processed (elapsed 37.1s)
  ndwi_std_year: cube groups 1975/1975 processed (elapsed 40.2s)
Finished ndwi_std_year in 42.7s

Merge completed.
Output written to: training_data_with_features_plus_monthly_indices.zarr


c:\Users\bartu\Desktop\Fonda-scikit\venv\Lib\site-packages\zarr\api\asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


## Post-merge validation

Run this after the merge cell finishes to verify schema and random value consistency against the source dataset.

In [5]:
merged_ds = xr.open_zarr(OUTPUT_PATH)

for feature_name in FEATURES_TO_MERGE:
    if feature_name not in merged_ds:
        raise KeyError(f"Missing merged feature in output: {feature_name}")

    da = merged_ds[feature_name]
    if tuple(da.dims) != ("pixel", "year"):
        raise ValueError(f"Feature {feature_name} dims are {da.dims}, expected ('pixel', 'year')")
    if da.dtype != np.float32:
        raise TypeError(f"Feature {feature_name} dtype is {da.dtype}, expected float32")

    nan_count = int(da.isnull().sum().compute().values)
    print(f"{feature_name}: shape={da.shape}, dtype={da.dtype}, nan_count={nan_count}")

rng = np.random.default_rng(42)
sampled_pixels = rng.integers(0, train_ds.sizes["pixel"], size=QA_SAMPLE_COUNT)
sampled_years = rng.integers(0, train_ds.sizes["year"], size=QA_SAMPLE_COUNT)

mismatches = []
for pixel_idx, year_idx in zip(sampled_pixels, sampled_years):
    source_cube_idx = int(mapped_source_cube_idx[pixel_idx])
    x_value = int(x_idx[pixel_idx])
    y_value = int(y_idx[pixel_idx])

    for feature_name in FEATURES_TO_MERGE:
        expected_value = (
            source_ds[feature_name]
            .isel(
                cube=source_cube_idx,
                year=int(year_idx),
                y=y_value,
                x=x_value,
            )
            .compute()
            .values
            .item()
        )

        actual_value = (
            merged_ds[feature_name]
            .isel(
                pixel=int(pixel_idx),
                year=int(year_idx),
            )
            .compute()
            .values
            .item()
        )

        if not np.isclose(actual_value, expected_value, equal_nan=True, rtol=1e-6, atol=1e-6):
            mismatches.append(
                {
                    "feature": feature_name,
                    "pixel": int(pixel_idx),
                    "year_index": int(year_idx),
                    "expected": float(expected_value) if not np.isnan(expected_value) else np.nan,
                    "actual": float(actual_value) if not np.isnan(actual_value) else np.nan,
                }
            )

if mismatches:
    raise AssertionError(f"Spot-check mismatches found: {mismatches[:5]}")

print(f"Random spot-check passed for {QA_SAMPLE_COUNT} sampled pixels.")

ndvi_cv_year: shape=(8155205, 7), dtype=float32, nan_count=3431893
ndvi_max_m2m_drop_year: shape=(8155205, 7), dtype=float32, nan_count=5674072
ndvi_max_year: shape=(8155205, 7), dtype=float32, nan_count=3431671
ndvi_min_year: shape=(8155205, 7), dtype=float32, nan_count=3431671
ndvi_std_year: shape=(8155205, 7), dtype=float32, nan_count=3431671
ndwi_cv_year: shape=(8155205, 7), dtype=float32, nan_count=3431882
ndwi_max_m2m_drop_year: shape=(8155205, 7), dtype=float32, nan_count=5674071
ndwi_max_year: shape=(8155205, 7), dtype=float32, nan_count=3431671
ndwi_min_year: shape=(8155205, 7), dtype=float32, nan_count=3431671
ndwi_std_year: shape=(8155205, 7), dtype=float32, nan_count=3431671
Random spot-check passed for 20 sampled pixels.


In [6]:
import pandas as pd

display_ds = xr.open_zarr(OUTPUT_PATH)

new_features = [feature for feature in FEATURES_TO_MERGE if feature in display_ds.data_vars]
missing_features = [feature for feature in FEATURES_TO_MERGE if feature not in display_ds.data_vars]

print("New features found in output dataset:")
for feature in new_features:
    da = display_ds[feature]
    print(f"- {feature}: dims={da.dims}, shape={da.shape}, dtype={da.dtype}")

if missing_features:
    print(f"\nMissing expected features: {missing_features}")

summary_df = pd.DataFrame(
    [
        {
            "feature": feature,
            "dims": str(tuple(display_ds[feature].dims)),
            "shape": str(tuple(display_ds[feature].shape)),
            "dtype": str(display_ds[feature].dtype),
        }
        for feature in new_features
    ]
)

summary_df

New features found in output dataset:
- ndvi_cv_year: dims=('pixel', 'year'), shape=(8155205, 7), dtype=float32
- ndvi_max_m2m_drop_year: dims=('pixel', 'year'), shape=(8155205, 7), dtype=float32
- ndvi_max_year: dims=('pixel', 'year'), shape=(8155205, 7), dtype=float32
- ndvi_min_year: dims=('pixel', 'year'), shape=(8155205, 7), dtype=float32
- ndvi_std_year: dims=('pixel', 'year'), shape=(8155205, 7), dtype=float32
- ndwi_cv_year: dims=('pixel', 'year'), shape=(8155205, 7), dtype=float32
- ndwi_max_m2m_drop_year: dims=('pixel', 'year'), shape=(8155205, 7), dtype=float32
- ndwi_max_year: dims=('pixel', 'year'), shape=(8155205, 7), dtype=float32
- ndwi_min_year: dims=('pixel', 'year'), shape=(8155205, 7), dtype=float32
- ndwi_std_year: dims=('pixel', 'year'), shape=(8155205, 7), dtype=float32


,feature,dims,shape,dtype
0,ndvi_cv_year,"('pixel', 'year')","(8155205, 7)",float32
1,ndvi_max_m2m_drop_year,"('pixel', 'year')","(8155205, 7)",float32
2,ndvi_max_year,"('pixel', 'year')","(8155205, 7)",float32
3,ndvi_min_year,"('pixel', 'year')","(8155205, 7)",float32
4,ndvi_std_year,"('pixel', 'year')","(8155205, 7)",float32
5,ndwi_cv_year,"('pixel', 'year')","(8155205, 7)",float32
6,ndwi_max_m2m_drop_year,"('pixel', 'year')","(8155205, 7)",float32
7,ndwi_max_year,"('pixel', 'year')","(8155205, 7)",float32
8,ndwi_min_year,"('pixel', 'year')","(8155205, 7)",float32
9,ndwi_std_year,"('pixel', 'year')","(8155205, 7)",float32
